# Practical 2: Feedforward Neural Network on MNIST (trained with SGD)

**What we are doing.** MNIST is 70,000 small (28×28) grayscale pictures of handwritten digits, 0–9. We **flatten** each picture into 784 numbers, pass them through two Dense layers, and use **softmax** to pick one of 10 digits. The problem statement says to train it with **SGD**.

```
picture 28x28 → Flatten 784 → Dense 128 (relu) → Dense 64 (relu) → Dense 10 (softmax) → digit
```
**Skeleton:** imports → load → preprocess → **model** → compile → fit → evaluate → plot/predict

> **Run locally in Jupyter?** Install once: `%pip install tensorflow numpy pandas matplotlib scikit-learn`, then *Kernel → Restart Kernel*.
> **In Colab:** *Runtime → Change runtime type → T4 GPU*. Everything is preinstalled.

## 1–3. Imports, load, preprocess
- `load_data()` gives training images/labels and test images/labels (`x` = images, `y` = correct answers).
- **Why ÷ 255?** Pixels go from 0 (black) to 255 (white). Dividing by the max puts them in 0–1, so training stays stable. This is called **normalization**.

In [ ]:
# 1. Imports
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# 2. Load data: 60k train / 10k test, 28x28 grayscale, labels 0-9
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 3. Preprocess: scale pixels 0-255 -> 0-1
x_train = x_train / 255.0
x_test = x_test / 255.0

## 4. Model (the block that changes between practicals)
- `Flatten`: 28×28 → 784, because a Dense layer only takes a flat list. It has no weights.
- `Dense(128, relu)`: 128 neurons, each looking at all 784 pixels. ReLU turns negatives into 0.
- `Dense(10, softmax)`: 10 probabilities that add up to 1, one per digit.
- Params of Dense(128) = 784 × 128 + 128 = **100,480**. Total = **109,386**.

In [ ]:
# 4. Model
model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),                         # 28x28 -> 784
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),   # one probability per digit
])
model.summary()

## 5–6. Compile and train
- **SGD** (learning rate 0.01, momentum 0.9), because the problem statement says *train using SGD*.
- **sparse_categorical_crossentropy**, because the labels are plain integers (7), not one-hot.
- 54,000 training images ÷ 128 per batch = **422 steps per epoch**. 10% (6,000) is held back for validation.

In [ ]:
# 5. Compile
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
              loss='sparse_categorical_crossentropy',   # labels are ints 0-9
              metrics=['accuracy'])

# 6. Train
history = model.fit(x_train, y_train, epochs=10, batch_size=128,
                    validation_split=0.1)

## 7–8. Evaluate and plot
Expect about **97–98% test accuracy**. Loss goes down and accuracy goes up.

In [ ]:
# 7. Evaluate on unseen test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test loss:", test_loss, " Test accuracy:", test_acc)

# 8. Plot loss and accuracy
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend()
plt.show()

---
## Understand it: poke cells
**A. What is the data?** An image is a 28×28 grid of numbers. After ÷255, 0 = black and 1 = white.

In [ ]:
import numpy as np
print(x_train.shape, y_train[:10])
plt.imshow(x_train[0], cmap='gray'); plt.title(y_train[0]); plt.show()
print(np.round(x_train[0][10:15, 10:15], 2))

**B. Flatten:** `(1, 28, 28)` becomes `(1, 784)`.

In [ ]:
print(x_train[:1].shape, layers.Flatten()(x_train[:1]).shape)

**C. ReLU and softmax on plain numbers.** ReLU: negatives → 0. Softmax: scores → probabilities that sum to 1.

In [ ]:
import tensorflow as tf
z = tf.constant([[-2.0, 0.0, 1.0, 3.0]])
print(tf.nn.relu(z).numpy())
print(tf.nn.softmax(z).numpy(), tf.reduce_sum(tf.nn.softmax(z)).numpy())

**D. One prediction:** 10 probabilities. `argmax` picks the biggest, and that's the predicted digit.

In [ ]:
p = model.predict(x_test[:1])
print(np.round(p, 3), "predicted:", p.argmax(), "actual:", y_test[0])

**E. Parameters per layer:** training adjusts these ~109k numbers.

In [ ]:
for l in model.layers: print(l.name, l.count_params())

---
## Break it: learning rate too big
With `learning_rate=1.0`, accuracy gets stuck at about **10%** (pure guessing among 10 digits). Each step overshoots, so it never settles: *learning rate too large → diverges*.

In [ ]:
bad = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
bad.compile(optimizer=keras.optimizers.SGD(learning_rate=1.0, momentum=0.9),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
bad.fit(x_train, y_train, epochs=2, batch_size=128, validation_split=0.1)

---
## Viva answers
**Q1. Why softmax with `sparse_categorical_crossentropy`?** Softmax gives one probability per class, summing to 1. *Sparse* because the labels are integers (7). The plain version expects one-hot labels ([0,…,1,…,0]).

**Q2. Parameters of Dense(128)?** 784 × 128 + 128 = **100,480**.

**Q3. Steps per epoch, and one step?** 54,000 ÷ 128 = **422**. One step: forward pass on 128 images → loss → backprop gradients → SGD update `w = w − lr × gradient`.

**Q4. Why ReLU?** It adds non-linearity, so stacked layers can learn complex patterns. It's cheap and avoids vanishing gradients.

**Q5. What is a feedforward network?** Data flows one way only, from input through hidden layers to output, with no loops.

> **Classmate's code** uses Adam, but the problem statement says *train using SGD*. Ours uses SGD.